# PySpark reference 5: dates, timestamps and time zones

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-reference-05-dates-and-times.ipynb)

Parsing and formatting, date parts, arithmetic, truncation, epochs, time-zone conversion, intervals, calendars, time buckets and session windows, plus the datetime pattern letters. Practise it in [PySpark practice 5: dates, timestamps and time zones](pyspark-practice-05-dates-and-times.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and small helpers: `plan_lines(df)` returns what `explain()` prints, `final_plan(df)` runs a query and returns the plan Spark actually used, and `expected_error()` prints the error class of an example that fails on purpose. The session time zone is UTC.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Date and timestamp functions](#date-and-timestamp-functions) | [`current_date`, `current_timestamp`](#current_date-current_timestamp), [`to_date`, `to_timestamp`](#to_date-to_timestamp), [`date_format`](#date_format), [`year`, `quarter`, `month`, `dayofmonth`](#year-quarter-month-dayofmonth), [`hour`, `minute`, `second`](#hour-minute-second), [`dayofweek`, `weekofyear`](#dayofweek-weekofyear), [`date_add`, `date_sub`, `add_months`](#date_add-date_sub-add_months), [`datediff`, `months_between`](#datediff-months_between), [`date_trunc`, `trunc`](#date_trunc-trunc), [`last_day`, `next_day`](#last_day-next_day), [`unix_timestamp`, `from_unixtime`](#unix_timestamp-from_unixtime), [`from_utc_timestamp`, `to_utc_timestamp`](#from_utc_timestamp-to_utc_timestamp), [`make_date`](#make_date), [`window`](#window) |
| [More date and time functions](#more-date-and-time-functions) | [`try_to_timestamp`](#try_to_timestamp), [`make_timestamp`, `try_make_timestamp`, `make_timestamp_ntz`](#make_timestamp-try_make_timestamp-make_timestamp_ntz), [`timestamp_diff`, `timestamp_add`](#timestamp_diff-timestamp_add), [`date_part`, `extract`](#date_part-extract), [`dayname`, `monthname`, `weekday`](#dayname-monthname-weekday), [`timestamp_seconds`, `timestamp_millis`, `unix_seconds`, `unix_millis`](#timestamp_seconds-timestamp_millis-unix_seconds-unix_millis), [`convert_timezone`](#convert_timezone), [Intervals: `make_interval`, `make_dt_interval` and interval arithmetic](#intervals-make_interval-make_dt_interval-and-interval-arithmetic), [Calendars with `sequence`](#calendars-with-sequence), [`session_window`](#session_window) |
| [Datetime patterns](#datetime-patterns) | [Pattern letters](#pattern-letters) |

## Date and timestamp functions

Parse, format, take apart and shift dates and timestamps. A `timestamp` is an instant shown in the session time
zone (UTC in this notebook); a `date` is a calendar day. Format letters are case-sensitive: `MM` is month, `mm`
is minute.

| function | what it does |
| --- | --- |
| `current_date`, `current_timestamp` | The date and instant when the query started, the same on every row of one query |
| `to_date`, `to_timestamp` | Parse text into a date or timestamp, with an optional pattern |
| `date_format` | Format a date or timestamp as text with a pattern |
| `year`, `quarter`, `month`, `dayofmonth` | Extract calendar parts as integers |
| `hour`, `minute`, `second` | Extract the time of day from a timestamp |
| `dayofweek`, `weekofyear` | Day of the week (Sunday = 1 to Saturday = 7) and ISO week number (weeks start on Monday) |
| `date_add`, `date_sub`, `add_months` | Move a date by days or months |
| `datediff`, `months_between` | Whole days between two dates (`end` first), and fractional months between two dates |
| `date_trunc`, `trunc` | Truncate to the start of a unit |
| `last_day`, `next_day` | The last day of the date's month, and the next given weekday strictly after the date |
| `unix_timestamp`, `from_unixtime` | Seconds since 1970-01-01 UTC, and back to formatted text |
| `from_utc_timestamp`, `to_utc_timestamp` | Convert between UTC and a named time zone's wall-clock time |
| `make_date` | Build a date from year, month and day columns |
| `window` | Assign each row to a fixed (or sliding) time bucket; returns a struct with `start` and `end` |

### `current_date`, `current_timestamp`

The date and instant when the query started, the same on every row of one query.

**Signature:** `F.current_date()` · `F.current_timestamp()`

**Use it for:** Audit columns such as `loaded_at`.

**Notes:** Results change on every run; pass the business date into a job instead of calling `current_date()`.

In [ ]:
spark.range(2).select("id", F.current_date().alias("today"), F.current_timestamp().alias("now")).printSchema()

### `to_date`, `to_timestamp`

Parse text into a date or timestamp, with an optional pattern.

**Signature:** `F.to_date(col, "pattern")` · `F.to_timestamp(col, "pattern")`

| parameter | type | description |
| --- | --- | --- |
| `col` | Column or name | string (or timestamp for `to_date`) |
| `format` | str, optional | a datetime pattern such as `dd/MM/yyyy HH:mm`; ISO 8601 when omitted |

**Use it for:** Typing raw dates such as `03/01/2026` or `2026-01-31 18:45`.

**Notes:** An input that does not match raises an error in ANSI mode; `F.try_to_timestamp` returns null.

In [ ]:
raw_dates = spark.createDataFrame([("03/01/2026", "2026-01-31 18:45")], "d string, ts string")
raw_dates.select(F.to_date("d", "dd/MM/yyyy").alias("to_date"), F.to_timestamp("ts", "yyyy-MM-dd HH:mm").alias("to_timestamp")).show()

### `date_format`

Format a date or timestamp as text with a pattern.

**Signature:** `F.date_format(col, "pattern")`

| parameter | type | description |
| --- | --- | --- |
| `date` | Column or name | date or timestamp |
| `format` | str | datetime pattern for the output text |

**Use it for:** Labels such as `2026-01` or `Monday`.

**Notes:** The result is a string, so it no longer sorts or compares as a date.

In [ ]:
orders.select("order_ts", F.date_format("order_ts", "yyyy-MM").alias("month"), F.date_format("order_ts", "EEEE").alias("weekday"),
              F.date_format("order_ts", "dd MMM HH:mm").alias("short")).show(3)

### `year`, `quarter`, `month`, `dayofmonth`

Extract calendar parts as integers.

**Signature:** `F.year(col)` · `F.quarter(col)` · `F.month(col)` · `F.dayofmonth(col)`

**Use it for:** Grouping and partition columns.

**Notes:** For grouping by month, `date_trunc` keeps one sortable value instead of two columns.

In [ ]:
orders.select("order_ts", F.year("order_ts").alias("year"), F.quarter("order_ts").alias("quarter"),
              F.month("order_ts").alias("month"), F.dayofmonth("order_ts").alias("day")).show(3)

### `hour`, `minute`, `second`

Extract the time of day from a timestamp.

**Signature:** `F.hour(col)` · `F.minute(col)` · `F.second(col)`

**Use it for:** Hour-of-day traffic patterns.

**Notes:** They read the time in the session time zone.

In [ ]:
orders.select("order_ts", F.hour("order_ts").alias("hour"), F.minute("order_ts").alias("minute"), F.second("order_ts").alias("second")).show(3)

### `dayofweek`, `weekofyear`

Day of the week (Sunday = 1 to Saturday = 7) and ISO week number (weeks start on Monday).

**Signature:** `F.dayofweek(col)` · `F.weekofyear(col)`

**Use it for:** Weekday reports and weekly grouping.

**Notes:** Near New Year the ISO week can belong to the previous or next year; group by `date_trunc("week", ts)` instead.

In [ ]:
orders.select("order_ts", F.date_format("order_ts", "E").alias("day"), F.dayofweek("order_ts").alias("dayofweek"),
              F.weekofyear("order_ts").alias("iso_week")).show(3)

### `date_add`, `date_sub`, `add_months`

Move a date by days or months.

**Signature:** `F.date_add(col, days)` · `F.date_sub(col, days)` · `F.add_months(col, months)`

**Use it for:** Due dates, look-back windows and month offsets.

**Notes:** Adding a month to 31 January lands on the last day of February.

In [ ]:
jan31 = spark.createDataFrame([(date(2026, 1, 31),)], "d date")
jan31.select("d", F.date_add("d", 1).alias("date_add"), F.date_sub("d", 31).alias("date_sub"), F.add_months("d", 1).alias("add_months")).show()

### `datediff`, `months_between`

Whole days between two dates (`end` first), and fractional months between two dates.

**Signature:** `F.datediff(end, start)` · `F.months_between(end, start)`

| parameter | type | description |
| --- | --- | --- |
| `end` | Column or name | later date |
| `start` | Column or name | earlier date |
| `roundOff` | bool, default True | `months_between`: round to 8 digits |

**Use it for:** Days to ship, customer age in months.

**Notes:** Swapping the arguments flips the sign.

In [ ]:
spans = spark.createDataFrame([(date(2026, 1, 5), date(2026, 3, 20))], "start date, end date")
spans.select(F.datediff("end", "start").alias("datediff"), F.datediff("start", "end").alias("swapped"),
             F.round(F.months_between("end", "start"), 2).alias("months_between")).show()

### `date_trunc`, `trunc`

Truncate to the start of a unit. `date_trunc(unit, ts)` returns a timestamp; `trunc(date, unit)` returns a date. Note the argument order.

**Signature:** `F.date_trunc("month", col)` · `F.trunc(col, "month")`

| parameter | type | description |
| --- | --- | --- |
| `format` | str | unit: `year`, `quarter`, `month`, `week`, `day`, `hour`, `minute`, `second` (`date_trunc` first argument, `trunc` second) |
| `timestamp / date` | Column or name | value to truncate |

**Use it for:** Grouping by week, month or quarter with one sortable key.

**Notes:** `date_trunc` takes the unit first and `trunc` takes it second; units include `"year"`, `"quarter"`, `"month"`, `"week"`, `"day"`, `"hour"`.

In [ ]:
orders.select("order_ts", F.date_trunc("week", "order_ts").alias("week_start"), F.date_trunc("hour", "order_ts").alias("hour_start"),
              F.trunc(F.to_date("order_ts"), "month").alias("month_start")).show(3)

### `last_day`, `next_day`

The last day of the date's month, and the next given weekday strictly after the date.

**Signature:** `F.last_day(col)` · `F.next_day(col, "Mon")`

**Use it for:** Month-end reporting and "next Monday" schedules.

**Notes:** `next_day` never returns the same day: for a Monday it returns the following Monday.

In [ ]:
orders.select(F.to_date("order_ts").alias("day"), F.last_day("order_ts").alias("last_day"), F.next_day(F.to_date("order_ts"), "Mon").alias("next_monday")).show(3)

### `unix_timestamp`, `from_unixtime`

Seconds since 1970-01-01 UTC, and back to formatted text.

**Signature:** `F.unix_timestamp(col)` · `F.from_unixtime(seconds, "pattern")`

**Use it for:** Systems that exchange epoch seconds.

**Notes:** For epoch milliseconds use `timestamp_millis` and `unix_millis`.

In [ ]:
orders.select("order_ts", F.unix_timestamp("order_ts").alias("epoch_s"),
              F.from_unixtime(F.unix_timestamp("order_ts"), "yyyy-MM-dd HH:mm").alias("from_unixtime")).show(3)

### `from_utc_timestamp`, `to_utc_timestamp`

Convert between UTC and a named time zone's wall-clock time.

**Signature:** `F.from_utc_timestamp(col, "America/Chicago")` · `F.to_utc_timestamp(col, "America/Chicago")`

| parameter | type | description |
| --- | --- | --- |
| `timestamp` | Column or name | timestamp |
| `tz` | str or Column | IANA zone such as `America/Chicago` |

**Use it for:** Business dates in a local time zone.

**Notes:** Use named zones, not fixed offsets, so daylight saving time is handled.

In [ ]:
orders.where("order_id = 12").select("order_ts", F.from_utc_timestamp("order_ts", "America/Chicago").alias("chicago_time"),
    F.to_utc_timestamp(F.from_utc_timestamp("order_ts", "America/Chicago"), "America/Chicago").alias("back_to_utc")).show()

### `make_date`

Build a date from year, month and day columns.

**Signature:** `F.make_date(year, month, day)`

**Use it for:** Sources that store dates as separate integers.

**Notes:** An impossible date such as 30 February raises an error in ANSI mode.

In [ ]:
spark.createDataFrame([(2026, 1, 31)], "y int, m int, d int").select(F.make_date("y", "m", "d").alias("make_date")).show()

### `window`

Assign each row to a fixed (or sliding) time bucket; returns a struct with `start` and `end`.

**Signature:** `F.window(ts, "1 day")` · `F.window(ts, "1 hour", "30 minutes")`

| parameter | type | description |
| --- | --- | --- |
| `timeColumn` | Column or name | timestamp column |
| `windowDuration` | str | bucket size: `10 minutes`, `1 hour`, `1 day` |
| `slideDuration` | str, optional | slide for overlapping windows |
| `startTime` | str, optional | offset of bucket boundaries, e.g. `15 minutes` |

**Use it for:** Counts per hour or per day, in batch and streaming.

**Notes:** Only buckets with rows appear; this grouping `window` is unrelated to `Window` for window functions.

In [ ]:
orders.groupBy(F.window("order_ts", "2 days")).agg(F.count("*").alias("orders")).select("window.start", "window.end", "orders").orderBy("start").show(3)

## More date and time functions

Parsing with fallbacks, building and differencing timestamps, epochs, time-zone conversion, intervals, calendars and sessions.

| function | what it does |
| --- | --- |
| `try_to_timestamp` | Parse text with an optional pattern, returning null instead of failing when the text does not match |
| `make_timestamp`, `try_make_timestamp`, `make_timestamp_ntz` | Build a timestamp from year, month, day, hour, minute and second columns, optionally in a given zone |
| `timestamp_diff`, `timestamp_add` | Whole units between two timestamps, and a timestamp moved by a number of units |
| `date_part`, `extract` | Extract a named field from a date, timestamp or interval: YEAR, QUARTER, MONTH, WEEK, DAY, DAYOFWEEK, DAYOFWEEK_ISO, DOY, HOUR, MINUTE, SECOND (with fraction) |
| `dayname`, `monthname`, `weekday` | Short English names of the day and month, and the weekday as 0 (Monday) to 6 (Sunday) |
| `timestamp_seconds`, `timestamp_millis`, `unix_seconds`, `unix_millis` | Convert between timestamps and epoch numbers (seconds or milliseconds since 1970-01-01 UTC) without string formatting |
| `convert_timezone` | Convert a wall-clock time from one zone to another zone's wall-clock time |
| Intervals: `make_interval`, `make_dt_interval` and interval arithmetic | Interval values add to and subtract from timestamps: `ts + make_dt_interval(days, hours, mins, secs)` or a SQL literal `INTERVAL 90 MINUTES` |
| Calendars with `sequence` | `F.sequence(start, stop, step)` builds an array of dates or timestamps; `explode` turns it into one row per day |
| `session_window` | Group events into sessions separated by a gap of inactivity; returns a struct with `start` and `end` where `end` is the last event plus the gap |

### `try_to_timestamp`

Parse text with an optional pattern, returning null instead of failing when the text does not match. There is no `try_to_date` in PySpark 4.0: wrap it as `to_date(try_to_timestamp(col, fmt))`.

**Signature:** `F.try_to_timestamp(col, F.lit(format))`

**Use it for:** Sources that mix several date formats: try each format and `coalesce` the results.

**Notes:** The format must be passed as a Column (`F.lit`).

In [ ]:
mixed = spark.createDataFrame([("2026-01-31",), ("31/01/2026",), ("01/31/2026",), ("2026-02-30",)], "text string")
mixed.select("text", F.to_date(F.coalesce(
    F.try_to_timestamp("text", F.lit("yyyy-MM-dd")),
    F.try_to_timestamp("text", F.lit("dd/MM/yyyy")),
)).alias("parsed")).show()

### `make_timestamp`, `try_make_timestamp`, `make_timestamp_ntz`

Build a timestamp from year, month, day, hour, minute and second columns, optionally in a given zone. `try_make_timestamp` returns null for impossible components such as 30 February.

**Signature:** `F.make_timestamp(years, months, days, hours, mins, secs, timezone=None)`

| parameter | type | description |
| --- | --- | --- |
| `years, months, days, hours, mins` | Column | integer components |
| `secs` | Column | seconds, may include a fraction |
| `timezone` | Column, optional | zone the components are in |

**Use it for:** Sources that store date and time parts separately.

**Notes:** Components are interpreted in `timezone`, or the session zone when omitted.

In [ ]:
parts = spark.createDataFrame([(2026, 1, 14, 0, 30, 0), (2026, 2, 30, 9, 0, 0)], "y int, mo int, d int, h int, mi int, s int")
parts.select(F.try_make_timestamp("y", "mo", "d", "h", "mi", "s").alias("utc"),
             F.try_make_timestamp("y", "mo", "d", "h", "mi", "s", F.lit("America/Chicago")).alias("chicago_wall_clock_as_utc")).show()
parts.limit(1).select(F.make_timestamp_ntz("y", "mo", "d", "h", "mi", "s").alias("wall_clock_ntz")).show()

### `timestamp_diff`, `timestamp_add`

Whole units between two timestamps, and a timestamp moved by a number of units. Units: YEAR, QUARTER, MONTH, WEEK, DAY, HOUR, MINUTE, SECOND, MILLISECOND, MICROSECOND.

**Signature:** `F.timestamp_diff("HOUR", start, end)` · `F.timestamp_add("MINUTE", F.lit(90), ts)`

| parameter | type | description |
| --- | --- | --- |
| `unit` | str | `YEAR`, `QUARTER`, `MONTH`, `WEEK`, `DAY`, `HOUR`, `MINUTE`, `SECOND`, `MILLISECOND`, `MICROSECOND` |
| `start, end / quantity, ts` | Column | timestamps to compare, or the amount to add and the timestamp |

**Use it for:** SLA durations (hours from placed to shipped) and deadlines.

**Notes:** The result is truncated toward zero: 1 hour 59 minutes is 1 HOUR. For fractional hours, subtract epoch seconds.

In [ ]:
span = spark.createDataFrame([(datetime(2026, 1, 5, 9, 15), datetime(2026, 1, 7, 8, 5))], "placed timestamp, shipped timestamp")
span.select(F.timestamp_diff("DAY", "placed", "shipped").alias("days"), F.timestamp_diff("HOUR", "placed", "shipped").alias("hours"),
            ((F.unix_timestamp("shipped") - F.unix_timestamp("placed")) / 3600).alias("exact_hours"),
            F.timestamp_add("HOUR", F.lit(48), "placed").alias("sla_deadline")).show()

### `date_part`, `extract`

Extract a named field from a date, timestamp or interval: YEAR, QUARTER, MONTH, WEEK, DAY, DAYOFWEEK, DAYOFWEEK_ISO, DOY, HOUR, MINUTE, SECOND (with fraction). For epoch seconds use `unix_seconds`.

**Signature:** `F.date_part(F.lit("DOY"), col)` · `F.extract(F.lit("HOUR"), col)`

**Use it for:** One generic function instead of many specific ones, matching SQL `EXTRACT(field FROM col)`.

**Notes:** The field name is a Column literal; SECOND returns a decimal including fractions.

In [ ]:
orders.where("order_id = 12").select("order_ts", F.date_part(F.lit("DOY"), "order_ts").alias("day_of_year"),
    F.extract(F.lit("DAYOFWEEK_ISO"), "order_ts").alias("iso_weekday"), F.extract(F.lit("SECOND"), "order_ts").alias("second")).show()

### `dayname`, `monthname`, `weekday`

Short English names of the day and month, and the weekday as 0 (Monday) to 6 (Sunday).

**Signature:** `F.dayname(col)` · `F.monthname(col)` · `F.weekday(col)`

**Use it for:** Labels without writing a `date_format` pattern.

**Notes:** `weekday` counts from Monday = 0, while `dayofweek` counts from Sunday = 1.

In [ ]:
orders.select("order_ts", F.dayname("order_ts").alias("dayname"), F.monthname("order_ts").alias("monthname"),
              F.weekday("order_ts").alias("weekday"), F.dayofweek("order_ts").alias("dayofweek")).show(3)

### `timestamp_seconds`, `timestamp_millis`, `unix_seconds`, `unix_millis`

Convert between timestamps and epoch numbers (seconds or milliseconds since 1970-01-01 UTC) without string formatting.

**Signature:** `F.timestamp_millis(col)` · `F.unix_millis(col)` (also `_seconds`, `_micros`)

**Use it for:** Kafka and JavaScript timestamps (milliseconds), Unix logs (seconds).

**Notes:** Check the unit: treating milliseconds as seconds gives dates around the year 58,000.

In [ ]:
epochs = spark.createDataFrame([(1768350600000,)], "ms bigint")
epochs.select("ms", F.timestamp_millis("ms").alias("from_millis"), F.timestamp_seconds(F.col("ms") / 1000).alias("from_seconds"),
              F.unix_millis(F.timestamp_millis("ms")).alias("back_to_millis")).show()

### `convert_timezone`

Convert a wall-clock time from one zone to another zone's wall-clock time. The input is a `timestamp_ntz` (a reading with no zone); the output is a `timestamp_ntz` in the target zone.

**Signature:** `F.convert_timezone(F.lit(source_zone), F.lit(target_zone), col)`

| parameter | type | description |
| --- | --- | --- |
| `sourceTz` | Column, optional | zone of the input wall-clock time (session zone if omitted) |
| `targetTz` | Column | zone to convert to |
| `sourceTs` | Column | a `timestamp_ntz` wall-clock value |

**Use it for:** Local schedules ("stores open at 09:00 local") that need to be compared across zones.

**Notes:** For event instants stored as `timestamp`, use `from_utc_timestamp` instead.

In [ ]:
local = spark.createDataFrame([(datetime(2026, 3, 8, 1, 30),), (datetime(2026, 3, 8, 3, 30),)], "store_time timestamp_ntz")
local.select("store_time", F.convert_timezone(F.lit("America/Chicago"), F.lit("UTC"), "store_time").alias("utc_wall_clock")).show()

### Intervals: `make_interval`, `make_dt_interval` and interval arithmetic

Interval values add to and subtract from timestamps: `ts + make_dt_interval(days, hours, mins, secs)` or a SQL literal `INTERVAL 90 MINUTES`. Subtracting two timestamps gives a day-time interval.

**Signature:** `F.make_dt_interval(days, hours, mins, secs)` · `F.expr("INTERVAL 2 HOURS")`

**Use it for:** Adding durations, and measuring elapsed time as an interval.

**Notes:** Month and year intervals (`make_ym_interval`) respect month lengths; day-time intervals are fixed durations.

In [ ]:
orders.where("order_id IN (1, 12)").select(
    "order_ts",
    (F.col("order_ts") + F.make_dt_interval(F.lit(0), F.lit(2), F.lit(30))).alias("plus_2h30"),
    (F.col("order_ts") + F.expr("INTERVAL 1 MONTH")).alias("plus_1_month"),
    (F.col("order_ts") - F.lit(datetime(2026, 1, 1)).cast("timestamp")).alias("since_new_year"),
).show(truncate=False)

### Calendars with `sequence`

`F.sequence(start, stop, step)` builds an array of dates or timestamps; `explode` turns it into one row per day. Use it to generate calendars and to find days with no data.

**Signature:** `F.sequence(start_date, end_date, F.expr("INTERVAL 1 DAY"))`

**Use it for:** Daily scaffolds for reports, gap detection, expanding date ranges into days.

**Notes:** The range is inclusive of both ends; large ranges explode into many rows.

In [ ]:
days = spark.range(1).select(F.explode(F.sequence(F.lit("2026-01-05").cast("date"), F.lit("2026-01-14").cast("date"))).alias("day"))
active = orders.select(F.to_date("order_ts").alias("day")).distinct()
days.join(active, "day", "left_anti").orderBy("day").show()

### `session_window`

Group events into sessions separated by a gap of inactivity; returns a struct with `start` and `end` where `end` is the last event plus the gap. Works in batch and in Structured Streaming.

**Signature:** `F.session_window(timeColumn, "30 minutes")`

| parameter | type | description |
| --- | --- | --- |
| `timeColumn` | Column or name | event time |
| `gapDuration` | str or Column | inactivity gap that closes a session, e.g. `30 minutes` |

**Use it for:** Web and app sessions without writing the lag-and-running-sum recipe by hand.

**Notes:** The gap is added after the last event, so `end - start` is the session length plus the gap.

In [ ]:
(events.dropDuplicates()
 .groupBy("user_id", F.session_window("ts", "30 minutes").alias("s"))
 .agg(F.count("*").alias("events"))
 .select("user_id", "s.start", "s.end", "events")
 .orderBy("user_id", "start").show(truncate=False))

## Datetime patterns

### Pattern letters

Patterns are used by `to_date`, `to_timestamp`, `try_to_timestamp`, `date_format` and `from_unixtime`. Letters are
**case-sensitive**; repeat a letter for width (`MM` → `01`, `MMM` → `Jan`, `MMMM` → `January`).

| letter | meaning | example |
| --- | --- | --- |
| `yyyy` / `yy` | year | 2026 / 26 |
| `MM` / `MMM` / `MMMM` | month | 01 / Jan / January |
| `dd` | day of month | 05 |
| `D` | day of year | 5 |
| `E` / `EEEE` | day name | Mon / Monday |
| `HH` | hour 0–23 | 18 |
| `hh` + `a` | hour 1–12 with AM/PM | 06 PM |
| `mm` | **minute** | 45 |
| `ss` | second | 07 |
| `SSS` | fraction of second | 120 |
| `Q` / `QQQ` | quarter | 1 / Q1 |
| `XXX` / `Z` / `VV` | zone offset or id | +01:00 / +0100 / Europe/Paris |
| `'text'` | literal text | `'T'` |

**Notes:** `YYYY` (week-based year) and `DD` (day of year) are common typos for `yyyy` and `dd`; Spark 4 rejects
some ambiguous legacy patterns with `INCONSISTENT_BEHAVIOR_CROSS_VERSION`.

In [ ]:
ts = spark.createDataFrame([(datetime(2026, 1, 5, 18, 45, 7),)], "ts timestamp")
ts.select(*[F.date_format("ts", p).alias(p) for p in ["yyyy-MM-dd", "dd MMM yyyy", "EEEE", "hh:mm a", "QQQ", "D", "yyyy-MM-dd'T'HH:mm:ssXXX"]]).show(truncate=False)

**Next:** practise these entries in [PySpark practice 5: dates, timestamps and time zones](pyspark-practice-05-dates-and-times.ipynb).